In [1]:
%load_ext autoreload
%autoreload 2
%env CUPY_ACCELERATORS=cutensor,cub

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
env: CUPY_ACCELERATORS=cutensor,cub


In [2]:
import tensorly as tl
import plotly.io as pio
#pio.renderers.default = 'iframe'
tl.set_backend('numpy')
tl.tenalg.set_backend('einsum')
tl.plugins.use_opt_einsum()
print(f'TensorLy backend: {tl.get_backend()}')

TensorLy backend: numpy


In [3]:
from moabb.datasets import *
from moabb.paradigms import P300

tmin=-0.2
dataset = BNCI2014_008()
paradigm = P300(tmin=-0.2)
epochs, y, meta = paradigm.get_data(dataset, return_epochs=True)
X = epochs.get_data()
groups=meta['subject']
X = tl.tensor(X)
X.shape

/usr/local/share/venv/lib/python3.12/site-packages/moabb/datasets/preprocessing.py:279: UserWarning: warnEpochs <Epochs | 4200 events (all good), -0.199 – 1 s (baseline off), ~79.0 MiB, data loaded,
 'Target': 700
 'NonTarget': 3500>
  warn(f"warnEpochs {epochs}")
/usr/local/share/venv/lib64/python3.12/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/usr/local/share/venv/lib/python3.12/site-packages/moabb/datasets/preprocessing.py:279: UserWarning: warnEpochs <Epochs | 4200 events (all good), -0.199 – 1 s (baseline off), ~79.0 MiB, data loaded,
 'Target': 700
 'NonTarget': 3500>
  warn(f"warnEpochs {epochs}")
/usr/local/share/venv/lib64/python3.12/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropr

Adding metadata with 3 columns
Adding metadata with 3 columns
Adding metadata with 3 columns
Adding metadata with 3 columns
Adding metadata with 3 columns
Adding metadata with 3 columns
Adding metadata with 3 columns
Adding metadata with 3 columns


/usr/local/share/venv/lib/python3.12/site-packages/moabb/datasets/preprocessing.py:279: UserWarning: warnEpochs <Epochs | 4200 events (all good), -0.199 – 1 s (baseline off), ~79.0 MiB, data loaded,
 'Target': 700
 'NonTarget': 3500>
  warn(f"warnEpochs {epochs}")
/usr/local/share/venv/lib64/python3.12/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/usr/local/share/venv/lib64/python3.12/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/usr/local/share/venv/lib64/python3.12/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline ins

Adding metadata with 3 columns
33600 matching events found
No baseline correction applied


(33600, 8, 308)

In [4]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.decomposition import PCA
from hoda.classification import SelectFCutoff
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

clf = make_pipeline(
    FunctionTransformer(tl.to_numpy),
    PCA(n_components=None, whiten=True),
    SelectFCutoff(cutoff=1),
    LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr')
)

In [5]:
from sklearn.model_selection import StratifiedGroupKFold
from hoda.classification import BTTDACV
from hoda.hoda import BTTDA

cv = StratifiedGroupKFold(n_splits=5)

hoda_params=  dict(
    max_iter=64,
    toeplitz=(1,),
    taper=False,
    verbose=True,
    refit_shrinkage=True,
)


bttdacv_params = dict(
    hoda_params=hoda_params,
    verbose=True,
    cv=cv,
    n_jobs=1,
    clf = clf,
)

bttdacv = BTTDACV(
    max_n_blocks=2,
    fixed_n_blocks=True,
    thetas=[0,0.1,0.2,0.3,0.4,0.5,0.6, 0.7, 0.8, 0.9,1],    
    **bttdacv_params
)

    

In [ ]:
bttdacv.fit(X,y, groups=groups)



fold=0, theta=0
Fitting block 1/2...


Forward model :   8%|██▏                        | 5/63 [00:00<00:06,  9.43it/s]


Fitting block 2/2...


Forward model :   8%|██▏                        | 5/63 [00:00<00:06,  9.51it/s]


fold=0, theta=0, n_blocks=2
fold=0, theta=0.1
Fitting block 1/2...


...

Forward model :  11%|███                        | 7/63 [00:01<00:11,  4.72it/s]


Fitting block 2/2...


Forward model :  17%|████▌                     | 11/63 [00:02<00:10,  5.00it/s]


fold=1, theta=0.6, n_blocks=2
fold=1, theta=0.7
Fitting block 1/2...


Backward HODA model rank=(1, 31):  97%|███████▊| 62/64 [00:42<00:01,  1.48it/s]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import plotly.express as px
from mne import EvokedArray
%matplotlib inline

plot_kwargs=dict(ts_args=dict(ylim=dict(eeg=[-2,2.5])))

target = X[y=='Target'].mean(axis=0)
non_target = X[y=='NonTarget'].mean(axis=0)
contrast = target - non_target
evoked = EvokedArray(tl.to_numpy(contrast), epochs.info, tmin=tmin)
evoked.plot_joint(**plot_kwargs)

X_prev = tl.zeros_like(X)
for bi,b in enumerate(bttdacv.blocks_):
    Xt = bttda.transform(X, n_blocks=bi+1)
    Xr = bttda.inv_transform(Xt, n_blocks=bi+1) - X_prev
    X_prev = Xr
    
    target = Xr[y=='Target'].mean(axis=0)
    non_target = Xr[y=='NonTarget'].mean(axis=0)
    contrast = target - non_target
    evoked = EvokedArray(tl.to_numpy(contrast), epochs.info, tmin=tmin)
    evoked.apply_baseline((-tmin, None))
    evoked.plot_joint(**plot_kwargs)